In [2]:
import pandas as pd

In [4]:
job_df = pd.read_csv("job_title_des.csv")

print("Dataset shape:", job_df.shape)
print("Columns:", job_df.columns.tolist())

job_df.head()

Dataset shape: (2277, 3)
Columns: ['Unnamed: 0', 'Job Title', 'Job Description']


,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [5]:
job_df.info()
print("\nMissing values:")
print(job_df.isnull().sum())

print("\nFirst 3 jobs:")
job_df.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2277 entries, 0 to 2276
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       2277 non-null   int64 
 1   Job Title        2277 non-null   object
 2   Job Description  2277 non-null   object
dtypes: int64(1), object(2)
memory usage: 53.5+ KB

Missing values:
Unnamed: 0         0
Job Title          0
Job Description    0
dtype: int64

First 3 jobs:


,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."


In [6]:
jobs_25 = job_df.head(25).copy()

print("Selected jobs:", jobs_25.shape)

Selected jobs: (25, 3)


In [7]:
print("JOB TITLE:")
print(jobs_25.iloc[0]["Job Title"])

print("\nJOB DESCRIPTION:")
print(jobs_25.iloc[0]["Job Description"])

JOB TITLE:
Flutter Developer

JOB DESCRIPTION:
We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.
Job Types: Full-time, Part-time
Salary: ₹20,000.00 - ₹40,000.00 per month
Benefits:
Flexible schedule
Food allowance
Schedule:
Day shift
Supplemental Pay:
Joining bonus
Overtime pay
Experience:
total work: 1 year (Preferred)
Housing rent subsidy:
Yes
Industry:
Software Development
Work Remotely:
Temporarily due to COVID-19


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

job_analysis_prompt = ChatPromptTemplate.from_template("""
You are a job posting analysis system.

Analyze the following job posting and perform FOUR tasks.

TASK 1 — Job Category
Classify the job into ONE broad category.

Use one of these categories:
- Technology/IT
- Finance
- Marketing
- Healthcare
- Education
- Sales
- Human Resources
- Operations
- Legal
- Other

TASK 2 — Required Skills and Technologies
Extract the important technical skills, tools, technologies,
programming languages, frameworks, platforms, and professional
skills explicitly required or strongly mentioned in the posting.

TASK 3 — Education
Identify the minimum or preferred educational qualification
mentioned in the job posting.

TASK 4 — Experience
Identify the required or preferred years/type of professional
experience mentioned in the job posting.

Return ONLY in this exact format:

Category: <one category>
Required Skills: <comma-separated skills, or None>
Education Required: <education requirement, or Not specified>
Experience Required: <experience requirement, or Not specified>

Do not provide any explanation outside this format.
Do not invent information that is not present in the job posting.

JOB TITLE:
{job_title}

JOB DESCRIPTION:
{job_description}
""")

In [10]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("LLM initialized successfully")

LLM initialized successfully


In [11]:
job_analysis_chain = job_analysis_prompt | llm | StrOutputParser()

In [12]:
test_job = jobs_25.iloc[0]

job_output = job_analysis_chain.invoke({
    "job_title": test_job["Job Title"],
    "job_description": test_job["Job Description"]
})

print(job_output)

Category: Technology/IT
Required Skills: Flutter
Education Required: Not specified
Experience Required: 1 year (Preferred)


In [13]:
def parse_job_analysis(output):
    result = {
        "Predicted_Category": "",
        "Required_Skills": "",
        "Education_Required": "",
        "Experience_Required": ""
    }

    for line in output.splitlines():
        line = line.strip()

        if line.startswith("Category:"):
            result["Predicted_Category"] = line.replace("Category:", "", 1).strip()

        elif line.startswith("Required Skills:"):
            result["Required_Skills"] = line.replace("Required Skills:", "", 1).strip()

        elif line.startswith("Education Required:"):
            result["Education_Required"] = line.replace("Education Required:", "", 1).strip()

        elif line.startswith("Experience Required:"):
            result["Experience_Required"] = line.replace("Experience Required:", "", 1).strip()

    return result

In [14]:
parsed_test = parse_job_analysis(job_output)

print(parsed_test)

{'Predicted_Category': 'Technology/IT', 'Required_Skills': 'Flutter', 'Education_Required': 'Not specified', 'Experience_Required': '1 year (Preferred)'}


In [15]:
import time

job_results = []

for index, row in jobs_25.iterrows():

    print(f"Processing job {index + 1}/25...")

    try:
        output = job_analysis_chain.invoke({
            "job_title": row["Job Title"],
            "job_description": row["Job Description"]
        })

        parsed = parse_job_analysis(output)
        job_results.append(parsed)

        # Small delay to reduce rate-limit problems
        time.sleep(2)

    except Exception as e:

        print(f"Error on job {index + 1}: {e}")

        job_results.append({
            "Predicted_Category": "ERROR",
            "Required_Skills": "ERROR",
            "Education_Required": "ERROR",
            "Experience_Required": "ERROR"
        })

        time.sleep(5)

print("\nProcessing completed!")

Processing job 1/25...
Processing job 2/25...
Processing job 3/25...
Processing job 4/25...
Processing job 5/25...
Processing job 6/25...
Processing job 7/25...
Processing job 8/25...
Processing job 9/25...
Processing job 10/25...
Processing job 11/25...
Processing job 12/25...
Processing job 13/25...
Processing job 14/25...
Processing job 15/25...
Processing job 16/25...
Processing job 17/25...
Processing job 18/25...
Processing job 19/25...
Processing job 20/25...
Processing job 21/25...
Processing job 22/25...
Processing job 23/25...
Processing job 24/25...
Processing job 25/25...

Processing completed!


In [16]:
job_results_df = pd.DataFrame(job_results)

print("AI results shape:", job_results_df.shape)
job_results_df.head()

AI results shape: (25, 4)


,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Technology/IT,Flutter,Not specified,1 year (Preferred)
1,Technology/IT,"Python, Django, Flask, REST, RPC, Linux, SQL, ...",Not specified,Not specified
2,Technology/IT,"Machine Learning, Deep Learning, Python, Java,...","Graduate or M.Sc. in Computer Science, Mathema...",Minimum 3 years of hands‑on development of com...
3,Technology/IT,"iOS development, Objective-C, Cocoa Touch, Cor...",Not specified,Not specified
4,Technology/IT,"React, React Native, JavaScript, HTML, CSS, RE...",Bachelor's degree in Computer Science or equiv...,"5+ years web development experience, including..."


In [17]:
final_jobs_df = pd.concat(
    [
        jobs_25.reset_index(drop=True),
        job_results_df.reset_index(drop=True)
    ],
    axis=1
)

print("Final shape:", final_jobs_df.shape)

Final shape: (25, 7)


In [18]:
final_jobs_df[
    [
        "Job Title",
        "Predicted_Category",
        "Required_Skills",
        "Education_Required",
        "Experience_Required"
    ]
].head(10)

,Job Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,Technology/IT,Flutter,Not specified,1 year (Preferred)
1,Django Developer,Technology/IT,"Python, Django, Flask, REST, RPC, Linux, SQL, ...",Not specified,Not specified
2,Machine Learning,Technology/IT,"Machine Learning, Deep Learning, Python, Java,...","Graduate or M.Sc. in Computer Science, Mathema...",Minimum 3 years of hands‑on development of com...
3,iOS Developer,Technology/IT,"iOS development, Objective-C, Cocoa Touch, Cor...",Not specified,Not specified
4,Full Stack Developer,Technology/IT,"React, React Native, JavaScript, HTML, CSS, RE...",Bachelor's degree in Computer Science or equiv...,"5+ years web development experience, including..."
5,Java Developer,Technology/IT,"C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQL...","Bachelor's Degree in Computer Science, Informa...",2 years of software development experience
6,Full Stack Developer,Technology/IT,"Node.js, Java, NoSQL, MongoDB, Elasticsearch, ...","B.Sc. degree in Computer Science, Engineering,...",Minimum 2 years of relevant development experi...
7,JavaScript Developer,Technology/IT,"ReactJS, NodeJS, Azure Functions, GraphQL, HTM...","Any graduation, Any PG, Any Doctorate",3-8 years
8,DevOps Engineer,Technology/IT,"Bash, Ruby, Python, Java, Puppet, Chef, Cloudi...",Not specified,Senior level
9,Software Engineer,Technology/IT,"REST API, C/C++, Linux/Unix, Python, Go, Git, ...","Minimum BS or MS in Computer Engineering, Comp...",Minimum 7 years of software development experi...


In [19]:
final_jobs_df.to_csv("part2_job_results.csv", index=False)

print("Part 2 results saved successfully!")

Part 2 results saved successfully!


In [20]:
import os

print("File exists:", os.path.exists("part2_job_results.csv"))
print("Rows:", len(final_jobs_df))
print("Columns:", final_jobs_df.columns.tolist())

File exists: True
Rows: 25
Columns: ['Unnamed: 0', 'Job Title', 'Job Description', 'Predicted_Category', 'Required_Skills', 'Education_Required', 'Experience_Required']


In [21]:
final_jobs_df[
    [
        "Job Title",
        "Predicted_Category",
        "Required_Skills",
        "Education_Required",
        "Experience_Required"
    ]
]

,Job Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,Technology/IT,Flutter,Not specified,1 year (Preferred)
1,Django Developer,Technology/IT,"Python, Django, Flask, REST, RPC, Linux, SQL, ...",Not specified,Not specified
2,Machine Learning,Technology/IT,"Machine Learning, Deep Learning, Python, Java,...","Graduate or M.Sc. in Computer Science, Mathema...",Minimum 3 years of hands‑on development of com...
3,iOS Developer,Technology/IT,"iOS development, Objective-C, Cocoa Touch, Cor...",Not specified,Not specified
4,Full Stack Developer,Technology/IT,"React, React Native, JavaScript, HTML, CSS, RE...",Bachelor's degree in Computer Science or equiv...,"5+ years web development experience, including..."
5,Java Developer,Technology/IT,"C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQL...","Bachelor's Degree in Computer Science, Informa...",2 years of software development experience
6,Full Stack Developer,Technology/IT,"Node.js, Java, NoSQL, MongoDB, Elasticsearch, ...","B.Sc. degree in Computer Science, Engineering,...",Minimum 2 years of relevant development experi...
7,JavaScript Developer,Technology/IT,"ReactJS, NodeJS, Azure Functions, GraphQL, HTM...","Any graduation, Any PG, Any Doctorate",3-8 years
8,DevOps Engineer,Technology/IT,"Bash, Ruby, Python, Java, Puppet, Chef, Cloudi...",Not specified,Senior level
9,Software Engineer,Technology/IT,"REST API, C/C++, Linux/Unix, Python, Go, Git, ...","Minimum BS or MS in Computer Engineering, Comp...",Minimum 7 years of software development experi...


In [23]:
print("df_30 exists:", "df_30" in globals())
print("results_df exists:", "results_df" in globals())

df_30 exists: False
results_df exists: False


In [26]:
import os

print(os.listdir())

['bbc-news-data.csv', 'Generative AI - Assignment 1.pdf', 'job_title_des.csv', 'part2_job_results.csv', 'part_1.ipynb', 'part_2.ipynb']


In [27]:
import os
import pandas as pd

print("Part 1 exists:", os.path.exists("part1_bbc_news_results.csv"))
print("Part 2 exists:", os.path.exists("part2_job_results.csv"))

part1 = pd.read_csv("part1_bbc_news_results.csv")
part2 = pd.read_csv("part2_job_results.csv")

print("\nPart 1 shape:", part1.shape)
print("Part 2 shape:", part2.shape)

Part 1 exists: True
Part 2 exists: True

Part 1 shape: (30, 11)
Part 2 shape: (25, 7)


In [28]:
part1[
    [
        "title",
        "category",
        "Detected_Topic",
        "Summary",
        "People",
        "Organizations",
        "Locations"
    ]
].head(5)

,title,category,Detected_Topic,Summary,People,Organizations,Locations
0,Ad sales boost Time Warner profit,business,Business,TimeWarner reported a 76% jump in fourth‑quart...,Richard Parsons,"TimeWarner, Google, AOL, Warner Bros, SEC, Ber...",United States
1,Dollar gains on Greenspan speech,business,Business,The dollar rose to its highest level against t...,"Alan Greenspan, Robert Sinche","Federal Reserve, Bank of America, G7, White House","New York, London, China"
2,Yukos unit buyer faces loan claim,business,Business,"Menatep Group, the owner of the former Yukos p...","Jamie Firestone, Tim Osborne, Mikhail Khodorko...","Yukos, Menatep Group, Rosneft, Yugansk","Russia, United States"
3,High fuel prices hit BA's profits,business,Business,British Airways reported a 40% fall in pre‑tax...,"Rod Eddington, Martin Broughton, Mike Powell, ...","British Airways, Dresdner Kleinwort Wasserstei...","United Kingdom, United States"
4,Pernod takeover talk lifts Domecq,business,Business,Shares in UK drinks and food group Allied Dome...,NaN,"Allied Domecq, Pernod Ricard, Wall Street Jour...","United Kingdom, France, London, Paris, Scotland"


In [29]:
part2[
    [
        "Job Title",
        "Predicted_Category",
        "Required_Skills",
        "Education_Required",
        "Experience_Required"
    ]
].head(5)

,Job Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,Technology/IT,Flutter,Not specified,1 year (Preferred)
1,Django Developer,Technology/IT,"Python, Django, Flask, REST, RPC, Linux, SQL, ...",Not specified,Not specified
2,Machine Learning,Technology/IT,"Machine Learning, Deep Learning, Python, Java,...","Graduate or M.Sc. in Computer Science, Mathema...",Minimum 3 years of hands‑on development of com...
3,iOS Developer,Technology/IT,"iOS development, Objective-C, Cocoa Touch, Cor...",Not specified,Not specified
4,Full Stack Developer,Technology/IT,"React, React Native, JavaScript, HTML, CSS, RE...",Bachelor's degree in Computer Science or equiv...,"5+ years web development experience, including..."


## Conclusion

In Part 2, a LangChain-based Generative AI pipeline was developed to analyze 25 job postings. The pipeline identifies the broad job category and extracts required skills/technologies, education requirements, and experience requirements.

The generated outputs were added to the original job posting data and saved as `part2_job_results.csv`.

Bonus////////////

In [31]:
import os
import time
import pandas as pd

job_all = pd.read_csv(
    "job_title_des.csv",
    engine="python",
    on_bad_lines="skip"
)

print("Total job postings:", len(job_all))

Total job postings: 2277


In [32]:
checkpoint_file = "bonus_part2_progress.csv"

if os.path.exists(checkpoint_file):
    job_saved_results = pd.read_csv(checkpoint_file)
    print("Existing checkpoint found:", len(job_saved_results), "jobs")
else:
    job_saved_results = pd.DataFrame(columns=[
        "Predicted_Category",
        "Required_Skills",
        "Education_Required",
        "Experience_Required"
    ])
    print("Starting Bonus Part 2 from scratch.")

Starting Bonus Part 2 from scratch.


In [33]:
start_index = len(job_saved_results)

print(f"Starting from job {start_index + 1} of {len(job_all)}")

for index in range(start_index, len(job_all)):

    row = job_all.iloc[index]

    print(f"Processing job {index + 1}/{len(job_all)}...")

    try:
        output = job_analysis_chain.invoke({
            "job_title": row["Job Title"],
            "job_description": row["Job Description"]
        })

        result = parse_job_analysis(output)

        job_saved_results = pd.concat(
            [job_saved_results, pd.DataFrame([result])],
            ignore_index=True
        )

        # Checkpoint every 10 jobs
        if (index + 1) % 10 == 0:
            job_saved_results.to_csv(
                checkpoint_file,
                index=False
            )
            print(f"Checkpoint saved: {index + 1} jobs")

        time.sleep(2)

    except Exception as e:

        print(f"Error at job {index + 1}: {e}")

        job_saved_results.to_csv(
            checkpoint_file,
            index=False
        )

        print("Progress saved. Stopping.")
        break

job_saved_results.to_csv(
    checkpoint_file,
    index=False
)

print("Current completed jobs:", len(job_saved_results))

Starting from job 1 of 2277
Processing job 1/2277...
Processing job 2/2277...
Processing job 3/2277...
Processing job 4/2277...
Processing job 5/2277...
Processing job 6/2277...
Processing job 7/2277...
Processing job 8/2277...
Processing job 9/2277...
Processing job 10/2277...
Checkpoint saved: 10 jobs
Processing job 11/2277...
Processing job 12/2277...
Processing job 13/2277...
Processing job 14/2277...
Processing job 15/2277...
Processing job 16/2277...
Processing job 17/2277...
Processing job 18/2277...
Processing job 19/2277...
Processing job 20/2277...
Checkpoint saved: 20 jobs
Processing job 21/2277...
Processing job 22/2277...
Processing job 23/2277...
Processing job 24/2277...
Processing job 25/2277...
Processing job 26/2277...
Processing job 27/2277...
Processing job 28/2277...
Processing job 29/2277...
Processing job 30/2277...
Checkpoint saved: 30 jobs
Processing job 31/2277...
Processing job 32/2277...
Processing job 33/2277...
Processing job 34/2277...
Processing job 35/2